In [ ]:
# ▶ 1. 필수 라이브러리 설치 및 임포트
!pip install openpyxl xlrd
import pandas as pd
import os
import re
import zipfile
from glob import glob

# ▶ 2. ZIP 압축 해제
def unzip_file(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
        print(f"✅ 압축 해제 완료: {extract_to}")

unzip_file("/content/14~23 주,야간 정리.zip", "/content/14_23")

# ▶ 3. 학과명 정제 함수
def clean_department_name(name):
    if pd.isna(name): return name
    name = str(name)
    name = re.sub(r"\(\d+\)", "", name)
    name = re.sub(r"[・ㆍ·‧․.•∙･]", "", name)
    name = name.replace(" ", "")
    return name.strip()

# ▶ 4. 연도별 계열 매핑 딕셔너리 생성
year_to_mapping = {}
for path in glob("/content/14_23/*.xls*"):
    try:
        df = pd.read_excel(path)
        if '학과명' in df.columns and '대계열' in df.columns and '조사년도' in df.columns:
            year = int(df['조사년도'].dropna().iloc[0])
            temp = df[['학과명', '대계열']].dropna().copy()
            temp['학과명'] = temp['학과명'].apply(clean_department_name)
            temp['대계열'] = temp['대계열'].astype(str).str.strip()
            mapping = dict(zip(temp['학과명'], temp['대계열']))

            if year in year_to_mapping:
                year_to_mapping[year].update(mapping)
            else:
                year_to_mapping[year] = mapping
    except Exception as e:
        print(f"❌ {os.path.basename(path)} → {e}")

# ▶ 5. 졸업생 진학률 원본 전처리 함수
def preprocess_graduate_file(file_path):
    df_raw = pd.read_excel(file_path, header=[0, 1])

    df_raw.columns = [
        ' '.join([str(c).strip() for c in col if 'Unnamed' not in str(c)]).strip()
        for col in df_raw.columns
    ]
    df = df_raw.copy()

    for col in ['기준연도', '학교명', '학과', '구분']:
        if col in df.columns:
            df[col] = df[col].ffill()

    df['학과'] = df['학과'].apply(clean_department_name)

    if '학교종류' in df.columns:
        df = df[~df['학교종류'].str.contains("사이버대학", na=False)]
    if '구분' in df.columns:
        df = df[~df['구분'].isin(['원격'])]

    # 연도 정수 변환
    df['기준년도'] = pd.to_numeric(df['기준연도'], errors='coerce')

    # 계열 매핑 함수 적용
    def match_by_year(row):
        year = int(row['기준년도']) if not pd.isna(row['기준년도']) else None
        dept = row['학과']
        mapping = year_to_mapping.get(year, {})
        return mapping.get(dept, None)

    df['계열'] = df.apply(match_by_year, axis=1)
    df['계열'] = df['계열'].fillna("미분류")

    return df

# ▶ 6. 파일 처리
file1 = '/content/2014~2020년도 졸업생 진학률.xlsx'
file2 = '/content/2021~2023 졸업생 진학률.xlsx'

df1 = preprocess_graduate_file(file1)
df2 = preprocess_graduate_file(file2)

# ▶ 7. 저장
df1.to_excel("/content/2014_2020_졸업생_계열분류.xlsx", index=False)
df2.to_excel("/content/2021_2023_졸업생_계열분류.xlsx", index=False)
print("✅ 졸업생 진학률 계열 분류 완료")


✅ 압축 해제 완료: /content/14_23
✅ 졸업생 진학률 계열 분류 완료


In [ ]:
import pandas as pd

# ▶ 1. 파일 로딩 (계열분류된 졸업생 진학률 데이터)
df1 = pd.read_excel("/content/2014_2020_졸업생_계열분류.xlsx")
df2 = pd.read_excel("/content/2021_2023_졸업생_계열분류.xlsx")
df = pd.concat([df1, df2], ignore_index=True)

# ▶ 2. 졸업자수/진학자수 계산
df['졸업자수'] = pd.to_numeric(df['졸업자(A)_남'], errors='coerce') + pd.to_numeric(df['졸업자(A)_여'], errors='coerce')
df['진학자수'] = pd.to_numeric(df['진학자(B)_남'], errors='coerce') + pd.to_numeric(df['진학자(B)_여'], errors='coerce')

# ▶ 3. 학과 열을 계열로 덮어쓰기 → 통합 기준용
df['학과'] = df['계열']

# ▶ 4. 통합 기준 및 집계 방식 정의
group_keys = ['기준연도', '학교명', '학과']
numeric_cols = ['졸업자수', '진학자수']
agg_dict = {col: 'sum' for col in numeric_cols}

# 나머지 정보 중 첫 번째 값만 유지할 항목 자동 탐색
first_cols = [col for col in df.columns if col not in group_keys + numeric_cols]
agg_dict.update({col: 'first' for col in first_cols})

# ▶ 5. 계열 기준 그룹 통합
grouped = df.groupby(group_keys, as_index=False).agg(agg_dict)

# ▶ 6. 진학률 계산
grouped['진학률(%)'] = (grouped['진학자수'] / grouped['졸업자수']) * 100
grouped['진학률(%)'] = grouped['진학률(%)'].round(2)

# ▶ 7. 열 순서 정렬
front_cols = group_keys + numeric_cols + ['진학률(%)']
remaining_cols = [col for col in grouped.columns if col not in front_cols]
final_cols = front_cols + remaining_cols
df_final = grouped[final_cols]

# ▶ 8. 저장
output_path = "/content/2014~2023_졸업생_진학률_계열통합_완성.xlsx"
df_final.to_excel(output_path, index=False)
print(f"✅ 계열 기준 진학률 계산 및 저장 완료 → {output_path}")


✅ 진학률 계산 및 저장 완료 → /content/2014~2023_졸업생_진학률_최종통합.xlsx
